# Synthetic Data Generation Pipeline
## Overview
In a production environment, data analyst rarely start with perfectly clean, structured datasets. This notebook will be used as the initial engineering phase of the project: **Data Architecture & Generation**

Using Python, Pandas, and synthetic data libraries (`Faker`), this script models and simulates a realistic 30-day clickstream event log and user database for a modern streetwear brand's recent themed apparel launch.

In [1]:
# Importing the libraries we need
from faker import Faker 
import pandas as pd
import numpy as np
import random
from datetime import timedelta

# Creating an instance of Faker
f = Faker(locale='en_US')

After initializing and installing the libraries we need, we can move on to generating a relational table of 10,000 unique users featuring randomized but realistic attributes (device types, acquisition traffic sources, and registration dates).

In [ ]:
# Creating a function to generate synthetic user  data
def generate_syn_users(num_of_users):
    # Creating an empty list to hold all users
    user_list = []
    
    # Creating a user dictionary
    for i in range(0, num_of_users):
        # Generating a random user
        user = {}
        user['user_id'] = i + 1
        user['join_date'] = f.date_time_between('-30d')
        user['device_type'] = f.random_element(elements=['Mobile', 'Desktop', 'Tablet'])
        user['traffic_source'] = f.random_element(elements=['Organic', 'Paid Social', 'Direct', 'Email'])
        user_list.append(user)
    
    return pd.DataFrame(user_list)

syn_users_df = generate_syn_users(10000)

# Sanity Check
syn_users_df.head()


,user_id,join_date,device_type,traffic_source
0,1,2026-05-17 03:25:55.149445,Desktop,Organic
1,2,2026-05-21 07:20:23.680912,Tablet,Organic
2,3,2026-06-02 17:22:22.481595,Desktop,Email
3,4,2026-05-19 22:21:07.343289,Tablet,Organic
4,5,2026-06-02 02:17:07.030461,Mobile,Direct


We have now successfully generated a relational table of 10,000 unique users featuring randomized but realistic attributes (device types, acquisition traffic sources, and registration dates). Now we will build a rule-based clickstream simulation enforcing a strict behavioral hierarchy:

* Landing Page View &rarr; Viewed Item &rarr; Added to Cart &rarr; Initiated Checkout &rarr; Completed Purchase 

We will mathematically embed sound drop-off rates at each stage of the funnel, including simulated friction points (such as high checkout abandonment among mobile users due to unexpected shipping costs).

In [ ]:
# Creating a function to generate synthetic event_log data
def generate_syn_event_logs(users_df):
    
    # Mock products that would be found in this store
    mock_products = [
        "Mecha Box Logo Hoodie",
        "Shinobi Compression Tee",
        "Akira Baggy Joggers",
        "Retro Nuptse-Style Puffer",
        "Neo-Tokyo Reflective Windbreaker",
        "EVA-01 Oversized Work Jacket",
        "Cyberpunk Cargo Pants",
        "Cursed Energy Knit Sweater",
        "Hokage Denim Jacket",
        "Shadow Clone Graphic Tee",
        "Chidri Track Pants",
        "Ghost in the Shell Balaclava",
        "Super Saiyan Heavyweight Hoodie",
        "Capsule Corp Canvas Tote",
        "Vagabond Distressed Bucket Hat"
    ]
    
    event_logs_data = []
    
    random.seed(42)
    np.random.seed(42)
    
    # Looping through df
    for row in users_df.itertuples():
        # Every user should start by landing_page_view, and we are setting up each user 
        user_events = ['landing_page_view']
        user_id = row.user_id
        current_time = row.join_date
        product_name = random.choice(mock_products)
        
        # 70% of users proceed to view a product
        if random.random() < 0.70:
            user_events.append('viewed_item')
            
            # 30% of users proceed to add the item to their cart
            if random.random() < 0.30:
                user_events.append('added_to_cart')
                
                # 50% of those who add to cart start checkout   
                if random.random() < 0.50:
                    user_events.append('initiated_checkout')
                    
                    # 40% of those who start checkout complete the purchase
                    if random.random() < 0.40:
                        user_events.append('completed_purchase')
        
        # Loading in the data into the df
        for stage in user_events:
            event_logs_data.append({
                'user_id': user_id, 
                'event_time': current_time,
                'event_type': stage,
                'product_name': product_name
            })
            # Creating mock time difference between event types
            current_time += timedelta(minutes=random.randint(1, 5), seconds=random.randint(0, 59))
            
    
    return pd.DataFrame(event_logs_data)

syn_event_logs_df = generate_syn_event_logs(syn_users_df)    

# Sanity check
syn_event_logs_df.head()

,user_id,event_time,event_type,product_name
0,1,2026-05-17 03:25:55.149445,landing_page_view,Chidri Track Pants
1,1,2026-05-17 03:28:09.149445,viewed_item,Chidri Track Pants
2,2,2026-05-21 07:20:23.680912,landing_page_view,Shinobi Compression Tee
3,2,2026-05-21 07:22:00.680912,viewed_item,Shinobi Compression Tee
4,3,2026-06-02 17:22:22.481595,landing_page_view,Mecha Box Logo Hoodie


Now that we have successfully created our mock data and stored them into respective dataframes, we can move onto storing them into a CSV so we can move on The Core Funnel Analysis (SQL) portion of our project.

In [ ]:
# Exporting each df to csv files
syn_users_df.to_csv('users.csv', index=False)
syn_event_logs_df.to_csv('events.csv', index=False)